In [1]:
import torch
from torch import nn
import matplotlib.pyplot as plt
import numpy as np
import cv2
import os
import torchsummary
import json
import torchvision

In [2]:
# 1. Data loader

# 2. 모델 생성

# 3. 학습 - 검증(테스트)

In [3]:
# 1. Data loader
#
#

dataset_train = []
dataset_valid = []

for root, dirs, filenames in os.walk('eyes'):
    if 'train' in root:
        dataset = dataset_train
    elif 'valid' in root:
        dataset = dataset_valid
    else:
        continue
    
    for filename in filenames:
        first, last = os.path.splitext(filename)
        if last != '.jpg':
            continue
        
        path = root + '/' + filename
        image = cv2.imread(path)
        
        if filename.startswith('s_'):
            label = 1
        elif filename.startswith('w_'):
            label = 0                    
        else:
            continue
        
        dataset.append((image, label))

In [4]:
# JSON 데이터에서 눈의 좌표를 읽어서 16개짜리 숫자 배열로 정리하는 함수


def read_eyes_data(data):

    dst = np.zeros(16)
    
    alt = {'rx':'rs', 'lx':'ls',
        'rs':'rs', 'ls':'ls', 'right':'right', 'left':'left'}

    dst_index = { 'rs':0, 'ls':4, 'right':8, 'left':12 }

    #p = (N, 2) 행렬
    def sort_ud(p):
        return p[p[:, 1].argsort()].copy()
    def sort_lr(p):
        return p[p[:, 0].argsort()].copy()

    sorts = { 'rs':sort_ud, 'ls':sort_ud, 'right':sort_lr, 'left':sort_lr }

    for shape in data['shapes']:
        p = np.array(shape['points'])
        shape['label'] = alt[shape['label']]
        if shape['label'] not in dst_index:
            continue # 에러 처리
        sort_fn = sorts[shape['label']]
        p = sort_fn(p).flatten()
        i = dst_index[shape['label']]
        dst[i:i+4] = p
    
    dst = dst.reshape(-1, 2) / np.array([
        data['imageWidth'],
        data['imageHeight']
    ])
    
    return dst.flatten()


data = json.load(open(r'C:\study\CNN_Sleep_Detector\eyes\train\eyes_sun\eyes_sun\s_ladofa876.json'))
points = read_eyes_data(data)

In [5]:
image = cv2.imread(r'C:\study\CNN_Sleep_Detector\eyes\train\eyes_sun\eyes_sun\s_ladofa876.jpg')


def draw_points(image, points):
    
    colors = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (0, 255, 255)]
   
    H, W, _ = image.shape

    points = (points.reshape(4, 2, 2) * np.array([W, H])).astype(int)

    for pair, color in zip(points, colors):
        p1, p2 = pair.astype(int)
        cv2.line(image, p1, p2, color, 2)

draw_points(image, points)    
cv2.imshow('image', image)
cv2.waitKey(0)

-1

In [6]:
# 1. Data loader
#
#

dataset_train = []
dataset_valid = []

for root, dirs, filenames in os.walk('eyes'):
    if 'train' in root:
        dataset = dataset_train
    elif 'valid' in root:
        dataset = dataset_valid
    else:
        continue
    
    for filename in filenames:
        first, last = os.path.splitext(filename)
        if last != '.json':
            continue
        
        json_path = root + '/' + filename
        jpg_path  = root + '/' + first + '.jpg'
        
        
        #json -> points 
        data = json.load(open(json_path))
        points = read_eyes_data(data)
        points = torch.tensor(points).float()
        
        if filename[0] == 's':
            sleep = torch.tensor(1.)
        else:
            sleep = torch.tensor(0.)
        
                
        image = cv2.imread(jpg_path)
        image = cv2.resize(image, (224, 224))
        
        # HWC -> CHW , 0~255 -> 0~1
        image = torch.tensor(image).permute(2, 0, 1) / 255
        
        
        
        dataset.append((image, points, sleep))
        
len(dataset_train), len(dataset_valid)

(162, 25)

In [7]:
loader_train = torch.utils.data.DataLoader(dataset_train, batch_size=32, shuffle=True, drop_last=True)
loader_valid = torch.utils.data.DataLoader(dataset_valid, batch_size=32)

for x, yp, ys in loader_train:
    break

x.shape, yp.shape, ys.shape


(torch.Size([32, 3, 224, 224]), torch.Size([32, 16]), torch.Size([32]))

In [8]:
# 기존에 만들어진 모델 가져오기
# tranfer learning


#    RegNetY_400MF (사전학습된 백본)
#            ↓
#    특징 추출 (440차원 벡터)
#            ↓
#    ┌────┴────┐
#    ▼         ▼
#   눈 좌표    졸음여부
#   (16개)      (1개)


base_model = torchvision.models.regnet_y_400mf(weights = 'DEFAULT')

base_model.fc = nn.Linear(440, 16 * 1)


class Head(nn.Module):
    def __init__(self):
        super().__init__()
        self.lin_points = nn.Linear(440, 16)
        self.lin_sleep =  nn.Linear(440, 1)
      
            
    def forward(self, input):
        points = self.lin_points(input)
        points = torch.clip(points, 0, 1)
        
        sleep = self.lin_sleep(input)
        sleep = torch.sigmoid(sleep)
        
        return points, sleep

base_model.fc = Head()

base_model.cuda()
points, sleep = base_model(x.cuda())


class BgrSwap(nn.Module):
    def forward(self, input):
        return input[..., [2, 1, 0]]


model = nn.Sequential(
    # 전처리....
    BgrSwap(),
    torchvision.transforms.Normalize( mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225] ),    
    base_model
)

model.cuda()
points, sleep = model(x.cuda()) 
points.shape, sleep.shape


opt = torch.optim.Adam(model.parameters())
loss_fn_mse = nn.MSELoss()
loss_fn_bce = nn.BCELoss()

mse = loss_fn_mse(points, yp.cuda())
bce = loss_fn_bce(sleep.squeeze(), ys.cuda())

loss = mse + bce

loss.backward()




In [10]:

for epoch in range(100):
    model.train()
    
    total_loss_mse = 0
    total_loss_bce = 0
    total_acc = 0
    
    for step, (x, yp, ys) in enumerate(loader_train):
        points, sleep = model(x.cuda())
        mse = loss_fn_mse(points, yp.cuda())
        bce = loss_fn_bce(sleep.squeeze(), ys.cuda())
        loss = mse + bce
        
        opt.zero_grad()
        loss.backward()
        opt.step()

        total_loss_mse += mse.item()
        total_loss_bce += bce.item()

        acc = ((sleep.squeeze().cpu() > 0.5).float() == ys).float().mean()
        total_acc += acc.item()

        print('\repoch=%d  step=%d  mse=%.4f  bce=%.4f  acc=%.4f' %
              (epoch, step,
               total_loss_mse / (step + 1),
               total_loss_bce / (step + 1),
               total_acc      / (step + 1)), end='')
    print()
    
    
    model.eval()
    
    total_loss_mse = 0
    total_loss_bce = 0
    total_acc = 0
    
    for step, (x, yp, ys) in enumerate(loader_valid):
        with torch.no_grad():
            points, sleep = model(x.cuda())
        mse = loss_fn_mse(points, yp.cuda())
        bce = loss_fn_bce(sleep.squeeze(), ys.cuda())
        loss = mse + bce
            
        #opt.zero_grad()
        #loss.backward()
        #opt.step()

        total_loss_mse += mse.item()
        total_loss_bce += bce.item()

        acc = ((sleep.squeeze().cpu() > 0.5).float() == ys).float().mean()
        total_acc += acc.item()

        print('\r                                                            mse=%.4f  bce=%.4f  acc=%.4f' %
                (
                total_loss_mse / (step + 1),
                total_loss_bce / (step + 1),
                total_acc      / (step + 1)), end='')
    print()
    
    

epoch=0  step=4  mse=0.0076  bce=0.0023  acc=1.0000
                                                            mse=0.0101  bce=2.0629  acc=0.6000
epoch=1  step=4  mse=0.0076  bce=0.0019  acc=1.0000
                                                            mse=0.0117  bce=2.1090  acc=0.6000
epoch=2  step=4  mse=0.0070  bce=0.0005  acc=1.0000
                                                            mse=0.0093  bce=2.0812  acc=0.5600
epoch=3  step=4  mse=0.0070  bce=0.0007  acc=1.0000
                                                            mse=0.0111  bce=2.0851  acc=0.5200
epoch=4  step=4  mse=0.0072  bce=0.0015  acc=1.0000
                                                            mse=0.0095  bce=2.0754  acc=0.5200
epoch=5  step=4  mse=0.0069  bce=0.0020  acc=1.0000
                                                            mse=0.0103  bce=2.0948  acc=0.5200
epoch=6  step=4  mse=0.0078  bce=0.0018  acc=1.0000
                                                            mse=0.

In [ ]:
for (x, yp, ys) in loader_valid:
    break

model.eval()
with torch.no_grad():
    points, sleep = model(x.cuda())
points = points.cpu().numpy()
sleep = sleep.cpu().numpy()
    
images = (x.numpy() * 255).transpose(0, 2, 3, 1).astype(np.uint8)


for i in range(25):
    image = images[i].copy()
    point = points[i]
    draw_points(image, point)
    cv2.imshow('image', image)
    cv2.waitKey(0)

: 